<a href="https://colab.research.google.com/github/apurvapm/RL-Stock-Trading-Agent/blob/main/rl_stock_trader_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📈 RL Stock Trader v2 — Multi-Ticker Double Dueling DQN

Improvements over v1:
- **Multi-ticker training** — agent trains on 7 stocks simultaneously (AAPL, MSFT, GOOGL, SPY, AMZN, NVDA, TSLA)
- **Fixed epsilon decay** — agent actually converges to exploitation instead of staying ~70% random
- **Trade penalty** — reward penalises unnecessary trades, killing the overtrading behaviour
- **More episodes** — 500 instead of 200

### Architecture
- **Dueling DQN** — separate value + advantage heads for stable Q-estimates
- **Double DQN** — decoupled action selection/evaluation to reduce overestimation
- **Experience Replay** — random minibatch sampling from a shared replay buffer
- **ε-greedy** with fast exponential decay (converges by ~ep 200)

### Action Space (5 actions)
| Action | Description |
|--------|-------------|
| 0 | Hold |
| 1 | Buy 50% of available cash |
| 2 | Sell 50% of holdings |
| 3 | Buy All |
| 4 | Sell All |

Run cells **top to bottom**. On Colab, run the install cell first.

## 1 · Install Dependencies

In [1]:
!pip install -q yfinance ta

  Preparing metadata (setup.py) ... done


## 2 · Imports

In [2]:
import random, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import deque
from dataclasses import dataclass, field
from typing import List, Dict, Tuple

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

try:
    import yfinance as yf
    HAS_YF = True
except ImportError:
    HAS_YF = False
    print('yfinance not found — using synthetic data.')

try:
    import ta
    HAS_TA = True
except ImportError:
    HAS_TA = False
    print('ta not found — using basic indicators only.')

## 3 · Config

**Key changes from v1:**
- `tickers` replaces single `ticker` — agent trains on all of them each run
- `epsilon_decay` dropped from 500 → 80 so the agent converges within 200 eps
- `episodes` raised from 200 → 500 for thorough training
- `trade_penalty` added — subtracted from reward on every non-Hold action

In [3]:
@dataclass
class Config:
    # Data — train on ALL of these simultaneously
    tickers:         List[str] = field(default_factory=lambda: [
        'AAPL', 'MSFT', 'GOOGL', 'SPY', 'AMZN', 'NVDA', 'TSLA'
    ])
    start_date:      str   = '2018-01-01'
    end_date:        str   = '2023-12-31'
    train_ratio:     float = 0.8

    # Environment
    initial_cash:    float = 10_000.0
    transaction_fee: float = 0.001    # 0.1% per trade
    trade_penalty:   float = 0.0005   # NEW: extra reward penalty per trade
    window_size:     int   = 20       # lookback steps

    # DQN
    hidden_size:     int   = 128
    lr:              float = 1e-3
    gamma:           float = 0.99
    epsilon_start:   float = 1.0
    epsilon_end:     float = 0.05
    epsilon_decay:   int   = 80       # FIX: was 500 — agent now converges by ~ep 200
    batch_size:      int   = 64
    memory_size:     int   = 50_000   # larger buffer for multi-ticker variety
    target_update:   int   = 10

    # Training
    episodes:        int   = 500      # more episodes now that we have 7 stocks
    seed:            int   = 42


CFG = Config()

random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Tickers: {CFG.tickers}')
print(f'epsilon_decay={CFG.epsilon_decay} → epsilon at ep 200: '
      f'{CFG.epsilon_end + (CFG.epsilon_start - CFG.epsilon_end) * math.exp(-200/CFG.epsilon_decay):.3f}')
print(f'epsilon at ep 500: '
      f'{CFG.epsilon_end + (CFG.epsilon_start - CFG.epsilon_end) * math.exp(-500/CFG.epsilon_decay):.3f}')

Device: cuda
Tickers: ['AAPL', 'MSFT', 'GOOGL', 'SPY', 'AMZN', 'NVDA', 'TSLA']
epsilon_decay=80 → epsilon at ep 200: 0.128
epsilon at ep 500: 0.052


## 4 · Data & Technical Indicators

Downloads OHLCV for **every ticker** in `CFG.tickers`. Falls back to synthetic GBM data for any ticker that fails to download.

In [4]:
def _synthetic_ohlcv(seed_offset: int = 0, n: int = 1500) -> pd.DataFrame:
    """Geometric Brownian Motion prices with fake volume."""
    np.random.seed(CFG.seed + seed_offset)
    ret   = np.random.normal(0.0003, 0.015, n)
    price = 100.0 * np.cumprod(1 + ret)
    noise = lambda s: np.random.uniform(0, s, n)
    close = price
    open_ = close * (1 + noise(0.005) - 0.0025)
    high  = np.maximum(close, open_) * (1 + noise(0.008))
    low   = np.minimum(close, open_) * (1 - noise(0.008))
    vol   = np.random.randint(500_000, 5_000_000, n)
    dates = pd.date_range('2018-01-01', periods=n, freq='B')
    return pd.DataFrame({'open': open_, 'high': high, 'low': low,
                          'close': close, 'volume': vol}, index=dates)


def add_indicators(df: pd.DataFrame) -> pd.DataFrame:
    c  = df['close']
    df = df.copy()
    df['ret_1']  = c.pct_change(1)
    df['ret_5']  = c.pct_change(5)
    df['ret_20'] = c.pct_change(20)
    df['vol_20'] = df['ret_1'].rolling(20).std()
    df['sma_10'] = c.rolling(10).mean() / c - 1
    df['sma_30'] = c.rolling(30).mean() / c - 1
    df['sma_50'] = c.rolling(50).mean() / c - 1
    delta = c.diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    df['rsi'] = 100 - 100 / (1 + gain / (loss + 1e-9))
    ema12 = c.ewm(span=12, adjust=False).mean()
    ema26 = c.ewm(span=26, adjust=False).mean()
    macd  = ema12 - ema26
    df['macd']        = macd / c
    df['macd_signal'] = macd.ewm(span=9, adjust=False).mean() / c
    mid = c.rolling(20).mean()
    std = c.rolling(20).std()
    df['bb_width'] = (2 * std) / (mid + 1e-9)
    df['bb_pos']   = (c - (mid - 2 * std)) / (4 * std + 1e-9)
    df['vol_norm'] = df['volume'] / df['volume'].rolling(20).mean()
    if HAS_TA:
        import ta as _ta
        df['atr'] = _ta.volatility.AverageTrueRange(
            df['high'], df['low'], df['close'], window=14).average_true_range() / c
        df['cci'] = _ta.trend.CCIIndicator(
            df['high'], df['low'], df['close'], window=20).cci() / 200
    return df


def fetch_one(ticker: str, i: int) -> pd.DataFrame:
    """Download a single ticker, fall back to synthetic on failure."""
    if HAS_YF:
        try:
            df = yf.download(ticker, start=CFG.start_date,
                             end=CFG.end_date, progress=False)
            df.columns = [c[0].lower() if isinstance(c, tuple) else c.lower()
                          for c in df.columns]
            df = df[['open', 'high', 'low', 'close', 'volume']].dropna()
            if df.empty:
                raise ValueError('empty')
        except Exception:
            print(f'  {ticker}: download failed — using synthetic data')
            df = _synthetic_ohlcv(seed_offset=i)
    else:
        df = _synthetic_ohlcv(seed_offset=i)
    df = add_indicators(df)
    df.dropna(inplace=True)
    return df


def fetch_all(cfg: Config) -> Dict[str, Tuple[pd.DataFrame, pd.DataFrame]]:
    """
    Download all tickers, add indicators, split train/test.
    Returns dict: {ticker: (train_df, test_df)}
    """
    result = {}
    print(f'Downloading {len(cfg.tickers)} tickers: {cfg.tickers}\n')
    for i, ticker in enumerate(cfg.tickers):
        df = fetch_one(ticker, i)
        split = int(len(df) * cfg.train_ratio)
        train_df = df.iloc[:split].reset_index(drop=True)
        test_df  = df.iloc[split:].reset_index(drop=True)
        result[ticker] = (train_df, test_df)
        print(f'  {ticker:6s}: {len(df):4d} rows total '
              f'({len(train_df)} train / {len(test_df)} test)')
    print(f'\nDone. {sum(len(v[0]) for v in result.values()):,} total training rows '
          f'across {len(result)} tickers.')
    return result


all_data = fetch_all(CFG)   # {ticker: (train_df, test_df)}
train_dfs = {t: v[0] for t, v in all_data.items()}
test_dfs  = {t: v[1] for t, v in all_data.items()}

/tmp/ipykernel_696/884750002.py:54: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=CFG.start_date,


  AAPL  : 1460 rows total (1168 train / 292 test)


/tmp/ipykernel_696/884750002.py:54: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=CFG.start_date,


  MSFT  : 1460 rows total (1168 train / 292 test)


/tmp/ipykernel_696/884750002.py:54: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=CFG.start_date,


...

  AMZN  : 1460 rows total (1168 train / 292 test)


/tmp/ipykernel_696/884750002.py:54: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=CFG.start_date,


  NVDA  : 1460 rows total (1168 train / 292 test)


/tmp/ipykernel_696/884750002.py:54: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=CFG.start_date,


  TSLA  : 1460 rows total (1168 train / 292 test)

Done. 8,176 total training rows across 7 tickers.


## 5 · Trading Environment

In [5]:
class StockEnv:
    """
    Single-stock trading environment.
    Actions: 0=Hold  1=Buy50%  2=Sell50%  3=BuyAll  4=SellAll

    v2 change: reward now subtracts trade_penalty on every non-Hold action
    to discourage excessive churning.
    """
    N_ACTIONS = 5

    def __init__(self, df: pd.DataFrame, cfg: Config):
        self.df        = df
        self.cfg       = cfg
        self.feat_cols = [c for c in df.columns
                          if c not in ('open', 'high', 'low', 'close', 'volume')]
        self.obs_size  = len(self.feat_cols) * cfg.window_size + 2
        self.reset()

    def reset(self):
        self.t       = self.cfg.window_size
        self.cash    = self.cfg.initial_cash
        self.shares  = 0.0
        self.trades  = 0
        self.history = []
        return self._obs()

    def step(self, action):
        price  = float(self.df['close'].iloc[self.t])
        before = self._portfolio(price)
        traded = self._execute(action, price)
        self.t += 1
        done   = self.t >= len(self.df) - 1
        after  = self._portfolio(float(self.df['close'].iloc[self.t]))
        reward = (after - before) / (before + 1e-9)
        # NEW: penalise trading to discourage churn
        if traded:
            reward -= self.cfg.trade_penalty
        self.history.append(after)
        return self._obs(), reward, done

    def _execute(self, action, price) -> bool:
        """Execute action, return True if a trade was made."""
        fee = self.cfg.transaction_fee
        if action == 1 and self.cash > 0:
            spend        = self.cash * 0.5
            self.shares += spend / (price * (1 + fee))
            self.cash   -= spend;  self.trades += 1;  return True
        elif action == 2 and self.shares > 0:
            sell         = self.shares * 0.5
            self.cash   += sell * price * (1 - fee)
            self.shares -= sell;   self.trades += 1;  return True
        elif action == 3 and self.cash > 0:
            self.shares += self.cash / (price * (1 + fee))
            self.cash    = 0.0;    self.trades += 1;  return True
        elif action == 4 and self.shares > 0:
            self.cash   += self.shares * price * (1 - fee)
            self.shares  = 0.0;    self.trades += 1;  return True
        return False

    def _portfolio(self, price):
        return self.cash + self.shares * price

    def _obs(self):
        w = self.df[self.feat_cols].iloc[
            self.t - self.cfg.window_size : self.t
        ].values.flatten().astype(np.float32)
        price = float(self.df['close'].iloc[self.t])
        total = self._portfolio(price) + 1e-9
        meta  = np.array([self.cash / total,
                           self.shares * price / total], dtype=np.float32)
        return np.nan_to_num(np.concatenate([w, meta]),
                             nan=0.0, posinf=1.0, neginf=-1.0)

    @property
    def final_value(self):
        return self._portfolio(float(self.df['close'].iloc[self.t]))


# Verify obs_size is consistent across all tickers (it should be)
_obs_sizes = {t: StockEnv(df, CFG).obs_size for t, df in train_dfs.items()}
print('Observation sizes per ticker:')
for t, s in _obs_sizes.items():
    print(f'  {t}: {s}')
assert len(set(_obs_sizes.values())) == 1, 'Obs sizes differ — all tickers must have the same features!'
OBS_SIZE = list(_obs_sizes.values())[0]
print(f'\nShared obs size: {OBS_SIZE}  |  Actions: {StockEnv.N_ACTIONS}')

Observation sizes per ticker:
  AAPL: 302
  MSFT: 302
  GOOGL: 302
  SPY: 302
  AMZN: 302
  NVDA: 302
  TSLA: 302

Shared obs size: 302  |  Actions: 5


## 6 · Dueling DQN Network

In [6]:
class DQN(nn.Module):
    """Dueling DQN with LayerNorm."""
    def __init__(self, obs_size, n_actions, hidden):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_size, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),   nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
        )
        self.value_head     = nn.Linear(hidden // 2, 1)
        self.advantage_head = nn.Linear(hidden // 2, n_actions)

    def forward(self, x):
        h = self.net(x)
        V = self.value_head(h)
        A = self.advantage_head(h)
        return V + A - A.mean(dim=-1, keepdim=True)


_net = DQN(OBS_SIZE, StockEnv.N_ACTIONS, CFG.hidden_size)
print(_net)
print(f'\nTotal parameters: {sum(p.numel() for p in _net.parameters()):,}')

DQN(
  (net): Sequential(
    (0): Linear(in_features=302, out_features=128, bias=True)
    (1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=128, bias=True)
    (4): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (5): ReLU()
    (6): Linear(in_features=128, out_features=64, bias=True)
    (7): ReLU()
  )
  (value_head): Linear(in_features=64, out_features=1, bias=True)
  (advantage_head): Linear(in_features=64, out_features=5, bias=True)
)

Total parameters: 64,454


## 7 · Replay Buffer & Agent

In [7]:
@dataclass
class Transition:
    state:      np.ndarray
    action:     int
    reward:     float
    next_state: np.ndarray
    done:       bool


class ReplayBuffer:
    def __init__(self, capacity):
        self.buf = deque(maxlen=capacity)
    def push(self, *args):
        self.buf.append(Transition(*args))
    def sample(self, n):
        return random.sample(self.buf, n)
    def __len__(self):
        return len(self.buf)


class DQNAgent:
    def __init__(self, obs_size, cfg):
        self.cfg    = cfg
        self.policy = DQN(obs_size, StockEnv.N_ACTIONS, cfg.hidden_size).to(DEVICE)
        self.target = DQN(obs_size, StockEnv.N_ACTIONS, cfg.hidden_size).to(DEVICE)
        self.target.load_state_dict(self.policy.state_dict())
        self.target.eval()
        self.optimizer = optim.Adam(self.policy.parameters(), lr=cfg.lr)
        self.memory    = ReplayBuffer(cfg.memory_size)
        self.steps     = 0
        self.episode   = 0

    @property
    def epsilon(self):
        return self.cfg.epsilon_end + (
            self.cfg.epsilon_start - self.cfg.epsilon_end
        ) * math.exp(-self.episode / self.cfg.epsilon_decay)

    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randrange(StockEnv.N_ACTIONS)
        with torch.no_grad():
            s = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)
            return int(self.policy(s).argmax(dim=1).item())

    def push(self, *args):
        self.memory.push(*args);  self.steps += 1

    def learn(self):
        if len(self.memory) < self.cfg.batch_size:
            return None
        batch   = self.memory.sample(self.cfg.batch_size)
        S  = torch.FloatTensor(np.array([t.state      for t in batch])).to(DEVICE)
        A  = torch.LongTensor( np.array([t.action     for t in batch])).to(DEVICE)
        R  = torch.FloatTensor(np.array([t.reward     for t in batch])).to(DEVICE)
        S2 = torch.FloatTensor(np.array([t.next_state for t in batch])).to(DEVICE)
        D  = torch.FloatTensor(np.array([t.done       for t in batch])).to(DEVICE)
        with torch.no_grad():
            next_a   = self.policy(S2).argmax(dim=1, keepdim=True)
            next_q   = self.target(S2).gather(1, next_a).squeeze()
            target_q = R + self.cfg.gamma * next_q * (1 - D)
        current_q = self.policy(S).gather(1, A.unsqueeze(1)).squeeze()
        loss = F.smooth_l1_loss(current_q, target_q)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.policy.parameters(), 1.0)
        self.optimizer.step()
        return loss.item()

    def sync_target(self):
        self.target.load_state_dict(self.policy.state_dict())

    def save(self, path='best_model_v2.pt'):
        torch.save(self.policy.state_dict(), path)
        print(f'Saved → {path}')

    def load(self, path='best_model_v2.pt'):
        self.policy.load_state_dict(torch.load(path, map_location=DEVICE))
        self.sync_target()


print('ReplayBuffer and DQNAgent defined.')

ReplayBuffer and DQNAgent defined.


## 8 · Training — Multi-Ticker

Each episode randomly picks one of the 7 training datasets. The agent sees a different
stock's price history every episode, which forces it to learn **general** trading patterns
rather than memorising one stock's specific moves.

Progress is printed every 50 episodes. The best checkpoint (by average return across a
random sample of tickers) is saved to `best_model_v2.pt`.

In [ ]:
agent      = DQNAgent(OBS_SIZE, CFG)
ticker_list = list(train_dfs.keys())

ep_returns, ep_losses, ep_epsilons, ep_tickers = [], [], [], []
best_avg_return = -np.inf

print(f'Training {CFG.episodes} episodes across {len(ticker_list)} tickers on {DEVICE}…')
print(f'epsilon will reach ~0.05 around episode {int(3 * CFG.epsilon_decay)}\n')

for ep in range(1, CFG.episodes + 1):
    # Pick a random ticker each episode
    ticker    = random.choice(ticker_list)
    train_env = StockEnv(train_dfs[ticker], CFG)
    state     = train_env.reset()
    total_loss, steps = 0.0, 0
    done = False

    while not done:
        action                = agent.select_action(state)
        next_state, rew, done = train_env.step(action)
        agent.push(state, action, rew, next_state, done)
        state = next_state
        loss  = agent.learn()
        if loss is not None:
            total_loss += loss;  steps += 1

    agent.episode += 1
    if ep % CFG.target_update == 0:
        agent.sync_target()

    ep_ret = (train_env.final_value - CFG.initial_cash) / CFG.initial_cash * 100
    ep_returns.append(ep_ret)
    ep_losses.append(total_loss / max(steps, 1))
    ep_epsilons.append(agent.epsilon)
    ep_tickers.append(ticker)

    # Every 50 eps, evaluate quickly on a random sample of all tickers
    # to get an unbiased checkpoint signal
    if ep % 50 == 0:
        sample_rets = []
        agent.policy.eval()
        for t in ticker_list:
            e = StockEnv(train_dfs[t], CFG)
            s = e.reset()
            d = False
            while not d:
                with torch.no_grad():
                    a = int(agent.policy(torch.FloatTensor(s).unsqueeze(0).to(DEVICE)).argmax(1).item())
                s, _, d = e.step(a)
            sample_rets.append((e.final_value - CFG.initial_cash) / CFG.initial_cash * 100)
        agent.policy.train()
        avg_ret = np.mean(sample_rets)
        if avg_ret > best_avg_return:
            best_avg_return = avg_ret
            agent.save('best_model_v2.pt')
        per_ticker = '  '.join(f'{t}:{r:+.0f}%' for t, r in zip(ticker_list, sample_rets))
        print(f'Ep {ep:>4}/{CFG.episodes} | ε={agent.epsilon:.3f} | '
              f'AvgTrain={avg_ret:+.1f}%  Best={best_avg_return:+.1f}%')
        print(f'         {per_ticker}')

print('\nLoading best model…')
agent.load('best_model_v2.pt')

Training 500 episodes across 7 tickers on cuda…
epsilon will reach ~0.05 around episode 240

Saved → best_model_v2.pt
Ep   50/500 | ε=0.558 | AvgTrain=+257.2%  Best=+257.2%
         AAPL:+268%  MSFT:+161%  GOOGL:+82%  SPY:+57%  AMZN:+41%  NVDA:+132%  TSLA:+1059%
Ep  100/500 | ε=0.322 | AvgTrain=+256.5%  Best=+257.2%
         AAPL:+267%  MSFT:+161%  GOOGL:+81%  SPY:+56%  AMZN:+41%  NVDA:+133%  TSLA:+1057%


## 9 · Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
ep = range(1, len(ep_returns) + 1)

ax = axes[0]
ax.plot(ep, ep_returns, alpha=0.2, color='steelblue', lw=1)
ax.plot(ep, pd.Series(ep_returns).rolling(30).mean(),
        color='steelblue', lw=2, label='30-ep MA')
ax.axhline(0, color='gray', ls='--', lw=0.8)
ax.set_title('Training Returns (%)'); ax.set_xlabel('Episode'); ax.legend()

ax = axes[1]
ax.plot(ep, ep_losses, alpha=0.3, color='crimson', lw=1)
ax.plot(ep, pd.Series(ep_losses).rolling(30).mean(), color='crimson', lw=2)
ax.set_title('Training Loss'); ax.set_xlabel('Episode')

ax = axes[2]
ax.plot(ep, ep_epsilons, color='darkorange', lw=2)
ax.axhline(CFG.epsilon_end, color='gray', ls='--', lw=0.8,
           label=f'ε_min={CFG.epsilon_end}')
ax.set_title('Epsilon Decay'); ax.set_xlabel('Episode')
ax.set_ylim(0, 1); ax.legend()

plt.suptitle(f'Training Curves — {len(ticker_list)} tickers', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves_v2.png', dpi=150, bbox_inches='tight')
plt.show()

## 10 · Evaluation on All Test Sets

Runs the trained agent with ε = 0 (pure exploitation) on the held-out test period
for **every ticker**. Also computes Buy & Hold for each as a benchmark.

In [ ]:
def evaluate_ticker(ticker: str, test_df: pd.DataFrame, agent: DQNAgent):
    """Run agent on one test set, return result dict."""
    agent.policy.eval()
    env    = StockEnv(test_df, CFG)
    state  = env.reset()
    action_log = []
    done   = False

    while not done:
        with torch.no_grad():
            s      = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)
            action = int(agent.policy(s).argmax(dim=1).item())
        state, _, done = env.step(action)
        action_log.append(action)

    prices   = test_df['close'].values[CFG.window_size:]
    bh_final = CFG.initial_cash * (prices[-1] / prices[0])
    bh_ret   = (bh_final - CFG.initial_cash) / CFG.initial_cash * 100

    rl_final = env.final_value
    rl_ret   = (rl_final - CFG.initial_cash) / CFG.initial_cash * 100

    ph       = np.array(env.history)
    dr       = np.diff(ph) / (ph[:-1] + 1e-9)
    sharpe   = (dr.mean() / (dr.std() + 1e-9)) * np.sqrt(252)
    peak     = np.maximum.accumulate(ph)
    max_dd   = ((ph - peak) / (peak + 1e-9)).min() * 100

    return dict(
        ticker=ticker, rl_final=rl_final, rl_ret=rl_ret,
        bh_final=bh_final, bh_ret=bh_ret,
        sharpe=sharpe, max_dd=max_dd, trades=env.trades,
        portfolio_hist=ph, prices=prices, action_log=action_log
    )


results = {t: evaluate_ticker(t, df, agent) for t, df in test_dfs.items()}

print('=' * 75)
print(f'  {"TICKER":<6}  {"RL Return":>10}  {"B&H Return":>10}  '
      f'{"Alpha":>8}  {"Sharpe":>7}  {"MaxDD":>8}  {"Trades":>7}')
print('=' * 75)
for r in results.values():
    alpha = r['rl_ret'] - r['bh_ret']
    print(f'  {r["ticker"]:<6}  {r["rl_ret"]:>+9.2f}%  {r["bh_ret"]:>+9.2f}%  '
          f'{alpha:>+7.2f}%  {r["sharpe"]:>7.3f}  {r["max_dd"]:>7.2f}%  {r["trades"]:>7d}')
print('=' * 75)
mean_alpha = np.mean([r['rl_ret'] - r['bh_ret'] for r in results.values()])
print(f'  Mean alpha vs B&H: {mean_alpha:+.2f}%')
print('=' * 75)

## 11 · Evaluation Plots — All Tickers

In [ ]:
n    = len(results)
cols = 3
rows = math.ceil(n / cols)
fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4 * rows))
axes = axes.flatten()

for idx, (ticker, r) in enumerate(results.items()):
    ax  = axes[idx]
    ph  = r['portfolio_hist']
    t   = range(len(ph))
    bh  = CFG.initial_cash * r['prices'] / r['prices'][0]
    ax.plot(t, ph, color='seagreen', lw=2, label=f'RL ({r["rl_ret"]:+.1f}%)')
    ax.plot(t, bh[:len(ph)], color='royalblue', lw=2, ls='--',
            label=f'B&H ({r["bh_ret"]:+.1f}%)')
    ax.axhline(CFG.initial_cash, color='gray', ls=':', lw=1)
    ax.set_title(ticker); ax.set_xlabel('Step'); ax.legend(fontsize=8)

for idx in range(n, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Portfolio Value — Test Set (all tickers)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('evaluation_all_tickers_v2.png', dpi=150, bbox_inches='tight')
plt.show()

# Action distribution summary
fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 3 * rows))
axes = axes.flatten()
labels = ['Hold', 'Buy 50%', 'Sell 50%', 'Buy All', 'Sell All']
colors = ['#aaaaaa', '#2ecc71', '#e74c3c', '#27ae60', '#c0392b']
for idx, (ticker, r) in enumerate(results.items()):
    ax     = axes[idx]
    counts = [r['action_log'].count(i) for i in range(StockEnv.N_ACTIONS)]
    bars   = ax.bar(labels, counts, color=colors, edgecolor='white')
    for bar, cnt in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                str(cnt), ha='center', va='bottom', fontsize=8)
    ax.set_title(f'{ticker} — {r["trades"]} trades')
    ax.tick_params(axis='x', rotation=25)

for idx in range(n, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Action Distribution — Test Set (all tickers)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('action_dist_all_tickers_v2.png', dpi=150, bbox_inches='tight')
plt.show()

## 12 · Save / Load Model

In [ ]:
# Best checkpoint already saved as best_model_v2.pt during training.
# Re-run this cell any time to explicitly save the current policy.
agent.save('best_model_v2.pt')

# To reload in a new session:
# new_agent = DQNAgent(OBS_SIZE, CFG)
# new_agent.load('best_model_v2.pt')

# To evaluate on a single ticker after reloading:
# r = evaluate_ticker('AAPL', test_dfs['AAPL'], new_agent)
# print(f'AAPL RL return: {r["rl_ret"]:+.2f}%')